# Lab 38: The False Memory Implant

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 5, *The Memory Hallucination*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-38.ipynb)

**What you will do:** test a friend for the DRM false-memory effect, compare their recognition answers with Roediger and McDermott's (1995) published data, explore a gist-based model of when a false memory crosses the "yes, I remember it" threshold, and check whether a small language model answers invented questions more accurately once it is given the source text to ground its answer.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 38 you read a list of fourteen sleep-associated words to a friend — bed, rest, awake,
tired, dream, wake, snooze, blanket, doze, slumber, nap, peace, yawn, drowsy — then asked
whether the word *sleep* was on the list. Most people say yes, confidently, even though
*sleep* was never read aloud. In the rigorous version (Step 5), you built a 20-word
recognition test mixing studied words, new words, and the critical lure *sleep*, and recorded
how often your friend said "old" ("this was on the list") to each kind of word.

## Record your results

Type your friend's answers from the Step 5 recognition test into the cell below. For each
category, enter how many words you tested and how many your friend called "old". If you
only ran the quick version (Steps 1–4), just fill in the single-lure row (1 tested, 1 if
they said yes to *sleep*) and leave the others at 0.

In [ ]:
# example data: replace with your friend's real recognition-test answers
category_n = {"studied (old)": 14, "critical lure (sleep)": 1,
              "weakly related new word": 3, "unrelated new word": 5}       # how many you tested
category_said_old = {"studied (old)": 12, "critical lure (sleep)": 1,
                      "weakly related new word": 1, "unrelated new word": 0}  # example data

results = pd.DataFrame({"tested": category_n, 'said "old"': category_said_old})
results['proportion "old"'] = (results['said "old"'] / results["tested"]).round(2)
display(results)

results['proportion "old"'].plot.barh(color=["steelblue", "firebrick", "orange", "grey"], figsize=(6, 3))
plt.xlim(0, 1); plt.xlabel('proportion judged "old"'); plt.gca().invert_yaxis(); plt.show()

## Compare

Figure 5.1 in the book reports Roediger and McDermott's (1995, Experiment 1, Table 1)
original recognition data: studied words were judged "old" .86 of the time, the critical
lure (*sleep*) .84 of the time, weakly related new words .21 of the time, and unrelated new
words only .02 of the time. In free recall, the critical lure was produced for 40% of the
lists even though it was never read aloud.

In [ ]:
benchmark = pd.Series({"studied (old)": 0.86, "critical lure (sleep)": 0.84,
                        "weakly related new word": 0.21, "unrelated new word": 0.02},
                       name="Roediger & McDermott (1995)")
comparison = results[['proportion "old"']].join(benchmark).rename(columns={'proportion "old"': "your friend"})
display(comparison)

comparison.plot.barh(figsize=(6, 3))
plt.xlim(0, 1); plt.xlabel('proportion judged "old"'); plt.gca().invert_yaxis(); plt.legend(loc="lower right"); plt.show()

lure_gap = comparison.loc["critical lure (sleep)", "your friend"] - comparison.loc["studied (old)", "your friend"]
print(f"Your friend's lure rate minus their hit rate: {lure_gap:+.2f}  "
      f"(Roediger & McDermott found -0.02: .84 vs .86 -- almost equal)")
if comparison.loc["critical lure (sleep)", "your friend"] >= comparison.loc["unrelated new word", "your friend"] + 0.3:
    print("The lure was judged 'old' far more often than a genuinely unrelated word, echoing the original result.")
else:
    print("With only one friend and one lure word, a single trial is noisy -- this is exactly why Roediger and")
    print("McDermott tested many participants across many lists before reporting the .84 figure.")

## Simulation

You probably "remembered" a word that was never presented. This simulation replicates the Deese-Roediger-McDermott (DRM) paradigm computationally: it creates word lists that are all semantically related to a critical lure, then models how a memory system based on gist (rather than verbatim recording) produces false memories.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# Simulated DRM list: the 14 studied words from Lab 38, all associated
# with the lure "SLEEP"
studied = ["bed", "rest", "awake", "tired", "dream", "wake", "snooze",
           "blanket", "doze", "slumber", "nap", "peace", "yawn", "drowsy"]
lure = "sleep"  # never presented

# Memory model: each word stored as "semantic activation" (0-1)
# Studied words get high activation; the lure gets activation only from
# spreading associations among the studied words (gist), not from being seen
studied_activation = np.random.uniform(0.6, 0.95, len(studied))
lure_activation = np.mean(studied_activation) * 0.85  # gist-based, not verbatim
unrelated_activation = np.random.uniform(0.05, 0.25, 5)  # random distractors

# Recognition threshold
threshold = 0.55

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(studied) + 1 + 5)
activations = np.concatenate([studied_activation, [lure_activation], unrelated_activation])
labels = studied + [f"{lure} (lure)"] + [f"distractor {i + 1}" for i in range(5)]
colours = ["steelblue"] * len(studied) + ["red"] + ["grey"] * 5

ax.bar(x, activations, color=colours)
ax.axhline(threshold, color="orange", linestyle="--", linewidth=2,
           label=f'"Yes" threshold ({threshold})')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=55, ha="right", fontsize=7)
ax.set_ylabel("Memory activation")
ax.set_title("DRM False Memory: The Lure Crosses the Threshold")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Lure activation: {lure_activation:.2f} (above threshold: {lure_activation > threshold})")
print("This is why you 'remember' a word that was never presented.")

**What to notice:** The red bar (the lure "sleep") crosses the recognition threshold even though it was never studied. Its activation comes entirely from its semantic overlap with the studied words. A parallel mechanism is at work in AI hallucinations—both systems generate plausible outputs from gist rather than retrieving verbatim records.

## Change the parameters

**gist_strength** controls how much of the studied words' average activation spreads to
the lure through pure semantic association (0.85 matches the simulation above).
**threshold** is the activation level needed for a "yes, I remember this" response. The
bars reuse the same `studied` word list from the cell above.
(If the sliders do not appear, run the cell again.)

In [ ]:
from ipywidgets import interact, FloatSlider

def explore(gist_strength=0.85, threshold=0.55):
    np.random.seed(3)
    studied_activation = np.random.uniform(0.6, 0.95, len(studied))
    lure_activation = np.mean(studied_activation) * gist_strength
    unrelated_activation = np.random.uniform(0.05, 0.25, 5)

    fig, ax = plt.subplots(figsize=(9, 4))
    x = np.arange(len(studied) + 1 + 5)
    activations = np.concatenate([studied_activation, [lure_activation], unrelated_activation])
    labels = studied + [f"{lure} (lure)"] + [f"distractor {i + 1}" for i in range(5)]
    colours = ["steelblue"] * len(studied) + ["red"] + ["grey"] * 5

    ax.bar(x, activations, color=colours)
    ax.axhline(threshold, color="orange", linestyle="--", linewidth=2, label=f'"Yes" threshold ({threshold:.2f})')
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=55, ha="right", fontsize=7)
    ax.set_ylabel("Memory activation")
    ax.set_title("DRM False Memory: Does the Lure Cross the Threshold?")
    ax.legend()
    plt.tight_layout()
    plt.show()

    verdict = "FALSE ALARM: the model 'remembers' sleep" if lure_activation > threshold else "correctly rejected"
    print(f"Lure activation: {lure_activation:.2f}  |  {verdict}")

interact(explore,
         gist_strength=FloatSlider(value=0.85, min=0.5, max=1.0, step=0.02),
         threshold=FloatSlider(value=0.55, min=0.3, max=0.9, step=0.02));

## The AI side

The book's AI Connection notes that retrieval-augmented generation (RAG) — grounding a
model's answer in a retrieved document rather than letting it generate purely from what it
"remembers" — is the AI analogue of checking your notes instead of trusting a hunch (Lewis
et al., 2020, *Advances in Neural Information Processing Systems*). Below, a small open
language model is asked about two facts it cannot know from training, because they were
invented for this notebook. First with no supporting text (it can only guess, like your
friend guessing "sleep"), then with the one sentence that actually states the fact placed
in its context (like handing it the source document, the way RAG does). We do not just look
at the generated text — we measure how much more probable the model finds the *correct*
answer once it can see the source.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
model = AutoModelForCausalLM.from_pretrained("distilgpt2")
model.eval()

def answer_logprob(prompt, answer):
    """Total log-probability the model assigns to `answer` as a continuation of `prompt`."""
    ids_prompt = tokenizer(prompt, return_tensors="pt").input_ids
    ids_full = tokenizer(prompt + answer, return_tensors="pt").input_ids
    with torch.no_grad():
        logits = model(ids_full).logits
    logprobs = torch.log_softmax(logits, dim=-1)
    total = 0.0
    for i in range(ids_prompt.shape[1], ids_full.shape[1]):
        total += logprobs[0, i - 1, ids_full[0, i]].item()
    return total

facts = [
    {"document": "The town of Bellhaven holds its annual kite festival in the month of October.",
     "question": "The town of Bellhaven holds its annual kite festival in the month of",
     "answer": " October"},
    {"document": "The fictional device the Glimmerwatch runs for 14 hours on a single charge.",
     "question": "The fictional device the Glimmerwatch runs for",
     "answer": " 14 hours"},
]

rows = []
for f in facts:
    no_context_lp = answer_logprob(f["question"], f["answer"])
    with_context_lp = answer_logprob(f["document"] + " " + f["question"], f["answer"])
    rows.append({"fact": f["document"][:45] + "...",
                 "log-prob, no source (from 'memory')": round(no_context_lp, 2),
                 "log-prob, source given (RAG-style)": round(with_context_lp, 2)})
ai = pd.DataFrame(rows)
display(ai)
gain = (ai["log-prob, source given (RAG-style)"] - ai["log-prob, no source (from 'memory')"]).mean()
print("Higher (less negative) log-probability means the model finds the correct answer more likely.")
print(f"Mean gain in log-probability from being given the source: {gain:+.2f}")

If the "source given" column is consistently higher, the model is far more confident in the
true answer once it can see the source text — exactly what retrieval-augmented generation
exploits. Without the source, the model still assigns *some* probability to the right
answer, and often to several plausible-sounding wrong ones too, because it is generating
from statistical patterns learned during training rather than a verified fact. The analogy
has a limit: your friend's false memory of "sleep" came from semantic association with
words that really were presented, so the false memory was *thematically justified*. The
model's guesses about Bellhaven or the Glimmerwatch, by contrast, come from generic
patterns in its training text that have nothing specifically to do with either invented
fact — it has no relevant "gist" to draw on at all. RAG fixes a model that has no relevant
knowledge; the DRM effect happens even when the studied material is genuinely relevant to
the false memory it produces.

## Pool the class data

One friend's answers are a single data point. Pool the class: each pair adds one row (an
anonymous pair ID, never names) recording whether the critical lure was judged "old" (1) or
correctly rejected (0), plus the hit rate on studied words. The example rows below are
placeholders so the cell runs.

In [ ]:
import io
csv_text = """pair_id,lure_judged_old,hit_rate_studied
G01,1,0.93
G02,1,0.86
G03,0,0.79
G04,1,1.00
G05,1,0.71
G06,0,0.93
"""  # example data: replace with your class CSV
df = pd.read_csv(io.StringIO(csv_text))
lure_rate = df.lure_judged_old.mean()
boot = [df.lure_judged_old.sample(len(df), replace=True).mean() for _ in range(5000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"Class false-alarm rate for the critical lure: {lure_rate:.0%} "
      f"(95% bootstrap interval {lo:.0%}-{hi:.0%}, n = {len(df)} pairs)")
print(f"Mean hit rate for studied words: {df.hit_rate_studied.mean():.0%}")
print("Roediger & McDermott (1995, Experiment 1) found .84 and .86 respectively.")

pd.Series({"your class: lure": lure_rate, "your class: hits": df.hit_rate_studied.mean(),
           "R&M 1995: lure": 0.84, "R&M 1995: hits": 0.86}).plot.barh(
    color=["firebrick", "steelblue", "lightcoral", "lightsteelblue"], figsize=(6, 3))
plt.xlim(0, 1); plt.xlabel('proportion judged "old"'); plt.show()

## Questions to think about

1. Lower gist_strength to about 0.5, which means much weaker semantic spreading. Does the lure still cross the threshold? What does this suggest about how strongly associated a false-memory lure needs to be before recognition memory is fooled?
2. Raise threshold high enough that some of the genuinely studied (blue) words also fall below it. What trade-off does this reveal about setting a stricter "yes" criterion in a real recognition test?
3. With gist_strength back at its default of 0.85, is the lure's bar ever clearly below the threshold? Given that Roediger and McDermott found false-alarm rates for the lure that matched or exceeded the hit rate for real words, does the default setting look realistic?
4. The lab explains false memory through a hippocampus-and-prefrontal-cortex system that stores gist rather than verbatim detail. Which slider in this widget is the better stand-in for "how much gist information has spread from the list to the lure" -- gist_strength or threshold? Explain your choice.

## Challenge

Run your own DRM demonstration on a friend, following Steps 1 to 5 of Lab 38 (or build a
new 8-10 word list around a different unstated lure). Record their actual recognition
answers and compare them with the model's prediction.

In [ ]:
# Example data: replace with your friend's real recognition-test answers.
my_lure_word = "sleep"                  # the never-presented word
friend_said_yes_to_lure = True          # example data
friend_hits = 12                        # example data: correct "yes" answers, out of 14 studied words
friend_false_alarms = 1                 # example data: "yes" answers to unrelated distractor words, out of 5

hit_rate = friend_hits / 14
print(f"Hit rate on real studied words: {hit_rate:.0%}")
print(f"Said yes to the never-presented lure '{my_lure_word}': {friend_said_yes_to_lure}")
print(f"False alarms on unrelated distractors: {friend_false_alarms} out of 5")
print("Compare this with Roediger and McDermott (1995), who found lure false-alarm rates")
print("that matched or exceeded the hit rate for words that were actually presented.")
print("The values above are example data -- replace them with your own experiment's results.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-38.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")